In [1]:
from ultralytics import YOLO
import cv2
import os

# Создание папки для сохранения кадров
output_folder = "saved_frames"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Загрузка модели YOLOv8x (самая точная версия)
model = YOLO("yolov8l.pt")

# Захват видеопотока
cap = cv2.VideoCapture("video_2025-01-19_21-33-09.mp4")

# Определение области интереса (ROI)
roi = [(130, 130), (600, 600)]  # Пример координат области

# Счетчик кадров
frame_count = 0

# Размер батча (количество кадров, обрабатываемых одновременно)
batch_size = 4

# Список для хранения кадров и их метаданных
frames = []
metadata = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Сжатие изображения на 20%
    height, width = frame.shape[:2]
    new_width = int(width * 0.8)
    new_height = int(height * 0.8)
    frame = cv2.resize(frame, (new_width, new_height))

    # Добавление кадра в батч
    frames.append(frame)
    metadata.append((new_height, new_width))

    # Если батч заполнен, обрабатываем его
    if len(frames) == batch_size:
        # Детекция объектов в батче
        results = model(frames)

        # Обработка результатов детекции для каждого кадра в батче
        for i, result in enumerate(results):
            frame = frames[i]
            height, width = metadata[i]

            # Обработка результатов детекции
            for box in result.boxes:
                if result.names[int(box.cls)] == "car":  # Проверка класса "car"
                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    # Проверка, что bounding box пересекается с ROI или полностью внутри
                    if (x1 < roi[1][0] and x2 > roi[0][0] and y1 < roi[1][1] and y2 > roi[0][1]) or \
                       (x1 >= roi[0][0] and y1 >= roi[0][1] and x2 <= roi[1][0] and y2 <= roi[1][1]):
                        # Увеличение счетчика кадров
                        frame_count += 1

                        # Обрезка кадра по ROI
                        roi_frame = frame[roi[0][1]:roi[1][1], roi[0][0]:roi[1][0]]

                        # Сохранение обрезанного кадра
                        output_path = os.path.join(output_folder, f"frame_{frame_count}.jpg")
                        cv2.imwrite(output_path, roi_frame)
                        print(f"Сохранен кадр: {output_path}")

            # Отображение кадра с выделенными объектами и ROI
            cv2.rectangle(frame, roi[0], roi[1], (0, 255, 0), 2)
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.imshow("Frame", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        # Очистка батча
        frames = []
        metadata = []

cap.release()
cv2.destroyAllWindows()

100%|██████████| 83.7M/83.7M [00:08<00:00, 10.2MB/s]



0: 640x384 (no detections), 233.5ms
1: 640x384 (no detections), 233.5ms
2: 640x384 (no detections), 233.5ms
3: 640x384 (no detections), 233.5ms
Speed: 1.5ms preprocess, 233.5ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 232.4ms
1: 640x384 (no detections), 232.4ms
2: 640x384 (no detections), 232.4ms
3: 640x384 2 suitcases, 232.4ms
Speed: 1.5ms preprocess, 232.4ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 219.9ms
1: 640x384 (no detections), 219.9ms
2: 640x384 (no detections), 219.9ms
3: 640x384 (no detections), 219.9ms
Speed: 1.3ms preprocess, 219.9ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 229.6ms
1: 640x384 2 suitcases, 229.6ms
2: 640x384 1 suitcase, 229.6ms
3: 640x384 1 suitcase, 229.6ms
Speed: 1.5ms preprocess, 229.6ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 suitcase, 222.0ms
1: 640x384 2 s